## Limpieza de 'DF_MYCHIP_SUCIO.csv'

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada evento) y le hacemos una primera inspección antes de limpiarla, siguiendo el mismo proceso que en buscametas/xipgroc/carreirasgalegas/ccnorte/cronofinisher.

`ciutat`/`comarca_regio` ya vienen directos de la fuente, y `distancia_m` ya es un número limpio en metros (no hay que parsear texto). `provincia_estat` sí necesita algo de trabajo: en realidad guarda la comunidad autónoma, no la provincia, así que la provincia real se saca geocodificando en reversa las coordenadas GPS de cada carrera.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/mychip/DF_MYCHIP_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (5150, 36)

ciutat                   object
comarca_regio            object
coord_1                 float64
coord_2                 float64
data                     object
distancia_m             float64
en_meta                 float64
en_meta_d                 int64
en_meta_h                 int64
en_meta_other           float64
esport                   object
estat_cursa              object
event_numeric_id          int64
event_slug               object
event_url                object
inscrits_d                int64
inscrits_h                int64
inscrits_other          float64
modalitat_id             object
modalitat_nom            object
no_presentats             int64
no_presentats_d           int64
no_presentats_h           int64
no_presentats_other     float64
nom_cursa                object
pais                     object
provincia_estat          object
retirats                float64
retirats_d                int64
retirats_h                int64
retirats_o

,ciutat,comarca_regio,coord_1,coord_2,data,distancia_m,en_meta,en_meta_d,en_meta_h,en_meta_other,...,provincia_estat,retirats,retirats_d,retirats_h,retirats_other,sortida_donada,sortida_donada_d,sortida_donada_h,sortida_donada_other,total_inscrits
0,Cullera,la Ribera Baixa,39.166099,-0.241100,2026-08-01T00:00:00Z,7000.0,326.0,72,254,NaN,...,Comunitat Valenciana,8.0,1,7,NaN,334,73,261,NaN,376
1,Cullera,la Ribera Baixa,39.166099,-0.241100,2026-08-01T00:00:00Z,7000.0,NaN,0,0,NaN,...,Comunitat Valenciana,NaN,0,0,NaN,1,0,0,NaN,1
2,Alfarp,NaN,39.276916,-0.560778,2026-07-04T00:00:00Z,7000.0,133.0,35,98,NaN,...,Valencian Community,1.0,1,0,NaN,134,36,98,NaN,198
3,Alfarp,NaN,39.276916,-0.560778,2026-07-04T00:00:00Z,7000.0,NaN,0,0,NaN,...,Valencian Community,NaN,0,0,NaN,0,0,0,NaN,0
4,la Vall d'Uixó,la Plana Baixa,39.823768,-0.245609,2026-06-25T00:00:00Z,10000.0,247.0,68,179,NaN,...,Comunitat Valenciana,2.0,0,2,NaN,249,68,181,NaN,281


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados, rango de fechas y
# consistencia de los recuentos por género.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print("Filas duplicadas por (event_numeric_id, modalitat_id):",
      curses.duplicated(subset=["event_numeric_id", "modalitat_id"]).sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())
print()

print("estat_cursa:")
print(curses["estat_cursa"].value_counts())
print()

print("en_meta vs en_meta_d + en_meta_h + en_meta_other, deberían coincidir:")
suma = curses["en_meta_d"].fillna(0) + curses["en_meta_h"].fillna(0) + curses["en_meta_other"].fillna(0)
print((curses["en_meta"].fillna(0) != suma).sum(), "filas no coinciden")
print()

print("Valores de 'esport':")
print(curses["esport"].value_counts())

Valores nulos por columna:
ciutat                    16
comarca_regio           1233
coord_1                    3
coord_2                    3
data                       0
distancia_m              701
en_meta                 1360
en_meta_d                  0
en_meta_h                  0
en_meta_other           4989
esport                     0
estat_cursa                0
event_numeric_id           0
event_slug                 0
event_url                  0
inscrits_d                 0
inscrits_h                 0
inscrits_other          4989
modalitat_id               0
modalitat_nom              0
no_presentats              0
no_presentats_d            0
no_presentats_h            0
no_presentats_other     4989
nom_cursa                  0
pais                       3
provincia_estat           15
retirats                3733
retirats_d                 0
retirats_h                 0
retirats_other          4989
sortida_donada             0
sortida_donada_d           0
sortida_donada_h

In [3]:
# Quitamos duplicados exactos, igual que en xipgroc/carreirasgalegas.
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

399 filas duplicadas eliminadas (5150 -> 4751)


In [4]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# "ciutat"/"comarca_regio"/"provincia_estat" ya vienen de la fuente (nada
# de geocodificar, a diferencia de xipgroc/ccnorte/carreirasgalegas/
# cronofinisher). "distancia_m" ya es un número limpio en metros. El resto
# de columnas (pais, coordenadas, event_slug/event_url, estat_cursa
# —siempre "finished_official"—, inscrits_*/no_presentats_*/retirats_*/
# sortida_donada_*) son artefactos técnicos o desgloses que no usamos en
# el resto de fuentes, así que no se incluyen.
curses_limpio = curses[
    ["nom_cursa", "data", "ciutat", "comarca_regio", "provincia_estat", "esport",
     "modalitat_nom", "distancia_m", "en_meta_d", "en_meta_h", "en_meta_other", "event_numeric_id"]
].rename(columns={
    "nom_cursa": "nombre_carrera",
    "data": "fecha",
    "ciutat": "municipio",
    "comarca_regio": "comarca",
    "provincia_estat": "provincia",
    "modalitat_nom": "modalidad",
    "en_meta_d": "finisher_d",
    "en_meta_h": "finisher_h",
    "en_meta_other": "finisher_desconocido",
    "event_numeric_id": "id",
})

# "fecha" viene en ISO con "Z" (UTC) — quitamos el huso horario para que
# el dtype sea igual que en el resto de fuentes (datetime64[ns] naive).
curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"]).dt.tz_localize(None)

curses_limpio["distancia"] = (curses_limpio["distancia_m"] / 1000).fillna(0)
curses_limpio = curses_limpio.drop(columns=["distancia_m"])

curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]] = (
    curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]].fillna(0).astype(int)
)

print(curses_limpio.shape)
curses_limpio.head()

(4751, 12)


,nombre_carrera,fecha,municipio,comarca,provincia,esport,modalidad,finisher_d,finisher_h,finisher_desconocido,id,distancia
0,XXIX CARRERA MATUTINA A LA PLAYA DE CULLERA,2026-08-01,Cullera,la Ribera Baixa,Comunitat Valenciana,running,7k,72,254,0,3387,7.0
1,XXIX CARRERA MATUTINA A LA PLAYA DE CULLERA,2026-08-01,Cullera,la Ribera Baixa,Comunitat Valenciana,running,7k + Camiseta,0,0,0,3387,7.0
2,V cursa Norcurna Alfarb,2026-07-04,Alfarp,NaN,Valencian Community,running,7K.,35,98,0,3381,7.0
3,V cursa Norcurna Alfarb,2026-07-04,Alfarp,NaN,Valencian Community,running,7k,0,0,0,3381,7.0
4,"12ª CARRERA NOCTURNA A LA FONT DE GARRUT, La V...",2026-06-25,la Vall d'Uixó,la Plana Baixa,Comunitat Valenciana,trail_running,Nocturna,68,179,0,3373,10.0


In [5]:
# "provincia" tal como llega de la fuente (antes "provincia_estat") es en
# realidad la comunidad autónoma (Comunitat Valenciana, Castilla-La
# Mancha...), no la provincia — por eso al cruzar con la población del
# INE en el notebook de análisis solo casaba un 3% de las filas. Aquí sí
# tenemos coordenadas GPS de cada carrera, así que la forma más fiable de
# sacar la provincia real es geocodificar en reversa esas coordenadas (a
# diferencia de geocodificar por nombre de ciudad, un punto GPS no tiene
# ambigüedad posible). Solo 356 ciudades únicas, así que es rápido.
# Checkpoint propio en mychip_ubicaciones.csv.
import csv
import time

coords_por_ciudad = (
    curses.dropna(subset=["coord_1", "coord_2"])
    .groupby("ciutat")[["coord_1", "coord_2"]].first()
)


def geocodificar_provincia_por_coordenadas(coords, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "mychip_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["ciutat"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} ciudades ya geocodificadas")

    geolocator = Nominatim(user_agent="mychip_ubicaciones_claudia")

    pendientes = [c for c in coords.index if c not in cache]
    print(f"Ciudades a geocodificar: {len(pendientes)} (de {len(coords)})")

    campos = ["ciutat", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, ciudad in enumerate(pendientes, 1):
            fila = {"ciutat": ciudad, "provincia": None}
            lat, lon = coords.loc[ciudad, "coord_1"], coords.loc[ciudad, "coord_2"]
            try:
                loc = geolocator.reverse((lat, lon), addressdetails=True, timeout=10)
                if loc:
                    # OSM da la provincia en "state_district", con el formato
                    # "nombre cooficial / nombre en castellano" cuando hay dos
                    # oficiales (p.ej. "València / Valencia") — nos quedamos
                    # con el segundo, que es el que usa el nomenclátor del INE.
                    state_district = loc.raw.get("address", {}).get("state_district")
                    if state_district:
                        fila["provincia"] = state_district.split("/")[-1].strip()
            except GeopyError as e:
                print(f"  [{ciudad}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{ciudad}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[ciudad] = fila

            if i % 50 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/mychip")
ubicaciones = geocodificar_provincia_por_coordenadas(coords_por_ciudad, out_dir=OUT_DIR)

provincia_real = curses_limpio["municipio"].map(lambda m: ubicaciones.get(m, {}).get("provincia"))
curses_limpio["comunidad_autonoma"] = curses_limpio["provincia"]
curses_limpio["provincia"] = provincia_real.fillna(curses_limpio["provincia"])

print("Filas con provincia real (antes solo teníamos la comunidad autónoma):",
      provincia_real.notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "comunidad_autonoma", "provincia"]].drop_duplicates("municipio").sample(15, random_state=0)


Checkpoint: 356 ciudades ya geocodificadas


Ciudades a geocodificar: 0 (de 356)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\mychip_data\mychip_ubicaciones.csv
Filas con provincia real (antes solo teníamos la comunidad autónoma): 4304 de 4751


,municipio,comunidad_autonoma,provincia
16,Crevillent,Comunitat Valenciana,Alicante
1129,Navarrés,Comunidad Valenciana,Valencia
3505,Letur,Castilla-La Mancha,Castilla-La Mancha
2136,Taradell,Cataluña,Barcelona
384,la Nucia,Comunitat Valenciana,Alicante
2327,Ollería,Comunidad Valenciana,Valencia
862,Alacant / Alicante,Comunitat Valenciana,Alicante
219,Vallada,Valencian Community,Valencia
129,Puçol,Comunitat Valenciana,Valencia
3432,Ador,Comunidad Valenciana,Valencia


In [6]:
# La disciplina ya viene dada directamente por "esport" (10 valores en
# inglés) — igual que "Modalitat" en xipgroc/"esport" en cronofinisher —
# así que no hace falta clasificarla por palabras clave. "swimming",
# "climbing" y "other" no encajan en ninguna categoría existente, van a
# "Otros".
_ESPORT_A_TIPO = {
    "trail_running": "trail running",
    "running": "road running",
    "btt": "Ciclismo y btt",
    "cycling": "Ciclismo y btt",
    "triathlon": "Multidisciplina",
    "walking": "marcha",
    "hiking": "marcha",
    "swimming": "Otros",
    "climbing": "Otros",
    "other": "Otros",
}
curses_limpio["tipo_modalidad"] = curses_limpio["esport"].map(_ESPORT_A_TIPO)
curses_limpio = curses_limpio.drop(columns=["esport"])

print(curses_limpio["tipo_modalidad"].value_counts())

tipo_modalidad
trail running      2306
road running       1163
Otros              1094
marcha               96
Ciclismo y btt       74
Multidisciplina      18
Name: count, dtype: int64


In [7]:
# Clasificamos el público (edad) por palabras clave, con SubXX llevando la
# edad directamente en el número (≤12 Infantil, 13-23 Cadete/Juvenil,
# igual que en xipgroc/cronofinisher). "Dorsal 0" (bib solidario/de
# honor) e "Invidentes"/"Handbikers"/discapacidad son categorías
# especiales, no de edad, así que van a "Otros". Si no hay ninguna marca
# de edad ni es especial, asumimos Absoluta/General por defecto.
import re

_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

def _clasificar_publico(row):
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    if re.search(_EQUIPOS_PATRON, texto_nombre):
        return "Equipos"

    texto = row["modalidad"]
    t = "" if pd.isna(texto) else texto.lower().strip()

    if re.search(_EQUIPOS_PATRON, t):
        return "Equipos"

    if re.search(r"discap|invident|handbike|silla de ruedas|adaptad|dorsal\s?0\b|dorsal cero", t):
        return "Otros"

    if re.search(r"\belit|profesional|\bpro\b", t):
        return "Elite"

    if re.search(r"veteran|master|m[aá]ster|\bsenior\b|\bsen\b", t):
        return "Mayores/Veteranos"

    _sb = re.search(r"\bsub\s?-?(\d{1,2})(?!\d)", t)
    if _sb:
        edad = int(_sb.group(1))
        if edad <= 12:
            return "Infantil"
        if edad <= 23:
            return "Cadete/Juvenil"

    if re.search(
        r"prebenjam|benjam|alev|infant|chupet|pitufo|querubin|menores|escolar|ni[nñ]os|a[nñ]os",
        t,
    ):
        return "Infantil"

    if re.search(r"cadete|juvenil|junior|j[uú]nior|promesa|preuniversitari", t):
        return "Cadete/Juvenil"

    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Texto de modalidad clasificado como Otros (categorías especiales):")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts())

publico
Absoluta/General     3686
Infantil              523
Cadete/Juvenil        433
Otros                  72
Equipos                16
Elite                  12
Mayores/Veteranos       9
Name: count, dtype: int64

Texto de modalidad clasificado como Otros (categorías especiales):
modalidad
Dorsal 0                                        25
Dorsal 0 (aportación 5€ solidarios)              7
Invidentes, FEDME                                6
dorsal 0                                         6
Invidentes, Trail                                5
Dorsal 0 a favor de NIÑOS CON CÁNCER AFANIÓN     3
Dorsal 0 (aportació 4€ solidaris)                2
Invidentes, FEMECV                               2
Dorsal 0, donativo voluntario                    2
Invidente                                        2
Dorsal 0, donatiu voluntari                      2
Invidentes                                       2
Invidentes Q20 FEMECV                            1
INVIDENTES FEDME                           

### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("mychip") que identifica de dónde viene cada fila al concatenar las 10 tablas. `municipio`/`comarca` ya venían directas de la fuente; `provincia` se ha recalculado geocodificando en reversa las coordenadas GPS (la que trae la fuente en origen es en realidad la comunidad autónoma, que se conserva aparte en `comunidad_autonoma`). Lo que es propio solo de mychip (`finisher_desconocido`, `id`, `modalidad`) va al final.

In [8]:
# Añadimos "fuente" (constante, identifica la plataforma de origen) y
# "dia_semana" (derivado de "fecha"), y reordenamos las columnas para que
# el esquema común quede igual en las 6 fuentes.
curses_limpio["fuente"] = "mychip"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconocido", "id", "modalidad", "comunidad_autonoma"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconocido',
 'id',
 'modalidad',
 'comunidad_autonoma']

In [9]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'finisher_desconocido', 'id', 'modalidad', 'comunidad_autonoma']
Filas x columnas: (4751, 16)

fuente                          object
nombre_carrera                  object
fecha                   datetime64[ns]
dia_semana                      object
distancia                      float64
tipo_modalidad                  object
publico                         object
finisher_d                       int64
finisher_h                       int64
municipio                       object
comarca                         object
provincia                       object
finisher_desconocido             int64
id                               int64
modalidad                       object
comunidad_autonoma              object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Cadete/Juvenil  Elite  Equipos

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,finisher_desconocido,id,modalidad,comunidad_autonoma
4550,mychip,"La Sagra SkyRace by Lurbel, Copa de España de ...",2013-09-15,Domingo,0.000,Otros,Absoluta/General,0,257,Puebla de Don Fadrique,NaN,Andalucía,0,112,COPA ESPANA,Andalucía
4675,mychip,Cross Comarcal Escolar y Carrera de Asfalto + ...,2012-11-11,Domingo,0.000,Otros,Infantil,23,24,Jávea,La Marina Alta,Alicante,0,60,PREBENJAMI,Comunidad Valenciana
4595,mychip,II Carrera por montaña Sierra de Callosa EXTREME,2013-05-05,Domingo,0.000,road running,Absoluta/General,3,99,Callosa de Segura,NaN,Alicante,0,86,default,Comunidad Valenciana
2196,mychip,1er Avenc trail Quatretonda,2020-03-01,Domingo,24.600,trail running,Absoluta/General,17,163,Cuatretonda,La Vall d'Albaida,Valencia,0,1453,Avenc trail,Comunidad Valenciana
1282,mychip,5k Massanassa per la igualtat 2023,2023-03-04,Sábado,5.000,road running,Absoluta/General,58,90,Masanasa,Huerta Sur,Valencia,4,1942,5k Corriendo,Comunidad Valenciana
2017,mychip,"ROTARY ALCOI, VOLTA ALS PONTS, PRO – ALZHEIMER",2021-06-12,Sábado,4.750,road running,Elite,0,0,Alcoy,La Hoya de Alcoy,Alicante,0,1622,Pro-Solidari,Comunidad Valenciana
3602,mychip,VI Carrera por montaña Cerro de la Mola,2017-01-15,Domingo,19.500,trail running,Absoluta/General,30,240,Novelda,El Vinalopó Medio,Alicante,0,691,CARRERA,Comunidad Valenciana
941,mychip,"I ULTRA MEDITERRÀNIA. TERRES DE TRAIL, Alcoi",2024-01-26,Viernes,46.000,trail running,Absoluta/General,31,256,Alcoi / Alcoy,l'Alcoià,Alicante,0,3030,46k Max,Comunitat Valenciana
3552,mychip,"Categories menors, XVII Volta a peu a Catadau",2017-02-26,Domingo,1.500,road running,Infantil,8,9,Catadau,La Ribera Alta,Valencia,0,763,"Infantil, 2004-2005",Comunidad Valenciana
4128,mychip,VII Volta a peu a la Vila D'Agullent,2015-06-06,Sábado,8.400,Otros,Absoluta/General,0,338,Agullent,La Vall d'Albaida,Valencia,0,397,HOMBRES,Comunidad Valenciana


In [10]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/mychip/DF_MYCHIP_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\mychip_data\DF_MYCHIP_LIMPIO.csv
